# 06 · Profile: crosswalks (SOC 2010→2018, OEWS hybrids, CIP 2010→2020, CIP→SOC)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Card: `docs/source-cards/crosswalks.md`.

**Why these sources matter.** The project's two ways in (brief S4) both end at a job (SOC 2018). Majors reach jobs only through the CIP→SOC crosswalk, and the job sources disagree on which job codes they use: older files use SOC 2010, OEWS 2019–20 and the projections use BLS hybrid / aggregate codes (`15-1256` …), and IPEDS 2015–19 uses CIP 2010. This notebook checks the bridges that make all of them line up:

| File | What it bridges |
|---|---|
| BLS `soc_structure_2018.xlsx` | The SOC 2018 hierarchy: which codes are real, and which detailed jobs sit under each group |
| BLS `soc_2010_to_2018_crosswalk.xlsx` | SOC 2010 → SOC 2018 |
| BLS `oes_2019_hybrid_structure.xlsx` | OEWS hybrid / aggregate codes → their SOC 2018 and SOC 2010 parts |
| NCES `Crosswalk2010to2020.csv` (+ `CIPCode2020.csv`) | CIP 2010 → CIP 2020 |
| NCES/BLS `CIP2020_SOC2018_Crosswalk.xlsx` | Major (CIP 2020) ↔ job (SOC 2018) |

**What this notebook does:** reads each file, checks its grain and special values, then (§6) runs every job code used by notebooks 02–04 and every major code in NYC's IPEDS files through the bridges, and lists what doesn't make it. It changes nothing.

Reading the eight OEWS workbooks (§6) takes about two minutes.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
SOC = r"\d\d-\d{4}"         # a 6-digit SOC-style code
CIP = r"\d\d\.\d{4}"        # a 6-digit (detailed) CIP code
NYC_COUNTIES = ["36005", "36047", "36061", "36081", "36085"]

## 0. Source header

| | **SOC structure + 2010→2018** | **OEWS hybrid structure** | **CIP 2010→2020** | **CIP 2020 ↔ SOC 2018** |
|---|---|---|---|---|
| **Publisher** | BLS, for the OMB SOC Policy Committee | BLS, OEWS program | NCES | NCES + BLS (joint) |
| **Edition** | SOC 2018 (final structure, Nov 2017) | Codes used for OEWS May 2019 and May 2020 | CIP 2020 | CIP 2020 × SOC 2018 |
| **How it was made** | Classification maintained by an interagency committee | BLS's own aggregation of SOC codes it can't yet estimate separately during the SOC 2010→2018 changeover | Classification review: codes kept, added, moved, deleted | **Expert judgment**: a match means the program "prepares people for" the job. Not based on where graduates actually work |
| **License** | U.S. government work, public domain | same | same | same |

Documentation: [SOC 2018](https://www.bls.gov/soc/2018/home.htm), [OEWS and the 2018 SOC](https://www.bls.gov/oes/soc_2018.htm), [CIP 2020](https://nces.ed.gov/ipeds/cipcode/), [CIP–SOC crosswalk](https://nces.ed.gov/ipeds/cipcode/resources.aspx?y=56).

In [2]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
KEYS = ["bls_soc_structure", "bls_soc_crosswalk", "bls_oews_hybrid", "nces_cip"]
FILES = man[man.key.isin(KEYS)][["key", "file", "downloaded_utc", "note"]].reset_index(drop=True)
FILES

,key,file,downloaded_utc,note
0,bls_soc_crosswalk,jobs/bls_soc_crosswalk/soc_2010_to_2018_crosswalk.xlsx,2026-10-12T17:05:25+00:00,Needed for OEWS 2018-2020 and older projections
1,nces_cip,graduates/nces_cip/Crosswalk2010to2020.csv,2026-10-12T17:08:43+00:00,CIP 2010 -> 2020 (IPEDS before C2020 uses CIP 2010)
2,nces_cip,graduates/nces_cip/CIPCode2020.csv,2026-10-12T17:08:44+00:00,
3,nces_cip,graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx,2026-10-12T17:08:46+00:00,Official major -> job crosswalk
4,bls_oews_hybrid,jobs/bls_oews_hybrid/oes_2019_hybrid_structure.xlsx,2026-10-13T00:35:44+00:00,OEWS May 2019-2020 hybrid codes and their SOC 2010 / 2018 parts
5,bls_soc_structure,jobs/bls_soc_structure/soc_structure_2018.xlsx,2026-10-13T00:36:51+00:00,"SOC 2018 hierarchy: major, minor, broad, detailed codes and titles"


## 1. Load as text

The BLS workbooks have title and note rows above the header, and the NCES CSVs wrap every code in an Excel formula (`="01.0101"`) so Excel keeps the leading zero. Read the raw sheets first to find the header row:

In [3]:
def peek(path, sheet=0, n=10):
    d = pd.read_excel(path, sheet_name=sheet, header=None, dtype=str, keep_default_na=False, nrows=n)
    return d.apply(lambda s: s.str[:45])

P_STRUCT = RAW / "jobs/bls_soc_structure/soc_structure_2018.xlsx"
P_SOCXW = RAW / "jobs/bls_soc_crosswalk/soc_2010_to_2018_crosswalk.xlsx"
P_HYB = RAW / "jobs/bls_oews_hybrid/oes_2019_hybrid_structure.xlsx"
P_CIPXW = RAW / "graduates/nces_cip/Crosswalk2010to2020.csv"
P_CIP20 = RAW / "graduates/nces_cip/CIPCode2020.csv"
P_CIPSOC = RAW / "graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx"

for p in [P_STRUCT, P_SOCXW, P_HYB, P_CIPSOC]:
    print(f"{p.name}: sheets {pd.ExcelFile(p).sheet_names}")
display(peek(P_SOCXW), peek(P_HYB, n=7))

soc_structure_2018.xlsx: sheets ['2018 Structure']
soc_2010_to_2018_crosswalk.xlsx: sheets ['Sorted by 2010']
oes_2019_hybrid_structure.xlsx: sheets ['OES2019 Hybrid', 'Field descriptions']
CIP2020_SOC2018_Crosswalk.xlsx: sheets ['File Guide', 'CIP-SOC', 'SOC-CIP', 'New CIP', 'New SOC', 'Added Matches', 'Unmatched CIP Codes', 'Unmatched SOC Codes']


,0,1,2,3
0,U.S. Bureau of Labor Statistics,,,
1,On behalf of the Office of Management and Bud,,,
2,,,,
3,November 2017,,,
4,,,,
5,***Questions should be emailed to soc@bls.gov,,,
6,,,,
7,,,,
8,2010 SOC Code,2010 SOC Title,2018 SOC Code,2018 SOC Title
9,11-1011,Chief Executives,11-1011,Chief Executives


,0,1,2,3,4,5,6,7,8
0,The table below shows how OES data collected,,,,,,,,
1,The first two columns show the code and title,,,,,,,,
2,"When the font is blue, the code and/or title",,,,,,,,
3,"When the font is green, the code and title ar",,,,,,,,
4,,,,,,,,,
5,OES 2019 Estimates Code,OES 2019 Estimates Title,2018 SOC Code,2018 SOC Title,OES 2018 Estimates Code,OES 2018 Estimates Title,2010 SOC Code,2010 SOC Title,NOTES
6,11-1011,Chief Executives,11-1011,Chief Executives,11-1011,Chief Executives,11-1011,Chief Executives,


In [4]:
# SOC 2018 structure: one column per level, the code sits in the column of its level
st = pd.read_excel(P_STRUCT, header=7, dtype=str, keep_default_na=False)
st.columns = ["major", "minor", "broad", "detailed", "title"]
st = st[st[["major", "minor", "broad", "detailed"]].apply(lambda r: "".join(r), axis=1).str.fullmatch(SOC)].copy()
st["level"] = st[["major", "minor", "broad", "detailed"]].ne("").idxmax(axis=1)
st["code"] = st[["major", "minor", "broad", "detailed"]].agg("".join, axis=1)
# parent chain: the file lists each code right after its parents, so carry the last code of each level forward
for lvl in ["major", "minor", "broad"]:
    st[f"in_{lvl}"] = st[lvl].replace("", pd.NA).ffill()
st = st.reset_index(drop=True)

socxw = pd.read_excel(P_SOCXW, header=8, dtype=str, keep_default_na=False)
socxw.columns = ["soc2010", "title2010", "soc2018", "title2018"]

hyb = pd.read_excel(P_HYB, header=5, dtype=str, keep_default_na=False)
hyb.columns = ["oes2019", "oes2019_title", "soc2018", "soc2018_title", "oes2018", "oes2018_title", "soc2010", "soc2010_title", "notes"]
hyb_padded = int((hyb != hyb.apply(lambda s: s.str.strip())).sum().sum())
hyb = hyb.apply(lambda s: s.str.strip())   # the hybrid file pads titles and notes; strip before any match

unwrap = lambda d: d.apply(lambda s: s.str.strip('="'))
cipxw = unwrap(pd.read_csv(P_CIPXW, dtype=str, keep_default_na=False))
cip20 = unwrap(pd.read_csv(P_CIP20, dtype=str, keep_default_na=False))
cipsoc = {s: pd.read_excel(P_CIPSOC, sheet_name=s, dtype=str, keep_default_na=False)
          for s in pd.ExcelFile(P_CIPSOC).sheet_names if s != "File Guide"}

shapes = {"SOC structure (code rows)": st, "SOC 2010→2018": socxw, "OEWS hybrid": hyb,
          "CIP 2010→2020": cipxw, "CIP 2020 list": cip20, **{f"CIP↔SOC: {k}": v for k, v in cipsoc.items()}}
display(pd.DataFrame({k: {"rows": len(v), "cols": v.shape[1]} for k, v in shapes.items()}).T)

texts = {"SOC 2010→2018": socxw, "CIP 2010→2020": cipxw, "CIP-SOC": cipsoc["CIP-SOC"]}
print("Cells with leading/trailing spaces:",
      {"OEWS hybrid (before strip)": hyb_padded, **{k: int((d != d.apply(lambda s: s.str.strip())).sum().sum()) for k, d in texts.items()}})
print("Padded hybrid cells that are codes:", int(sum((pd.read_excel(P_HYB, header=5, dtype=str, keep_default_na=False).iloc[:, i].pipe(lambda s: s != s.str.strip())).sum() for i in (0, 2, 4, 6))))
print("Titles carrying a footnote mark '(#'):",
      {"SOC 2010→2018": int(socxw.title2018.str.contains(r"\(#", regex=True).sum() + socxw.title2010.str.contains(r"\(#", regex=True).sum())})
print("Example:", socxw[socxw.title2018.str.contains(r"\(#", regex=True)].iloc[0].tolist())

,rows,cols
SOC structure (code rows),1447,10
SOC 2010→2018,900,4
OEWS hybrid,868,9
CIP 2010→2020,2699,6
CIP 2020 list,2848,8
CIP↔SOC: CIP-SOC,6097,4
CIP↔SOC: SOC-CIP,6093,4
CIP↔SOC: New CIP,1076,4
CIP↔SOC: New SOC,415,4
CIP↔SOC: Added Matches,1007,4


Cells with leading/trailing spaces: {'OEWS hybrid (before strip)': 889, 'SOC 2010→2018': 0, 'CIP 2010→2020': 0, 'CIP-SOC': 0}


Padded hybrid cells that are codes: 0
Titles carrying a footnote mark '(#'): {'SOC 2010→2018': 166}
Example: ['11-9199', 'Managers, All Other (#)', '13-1082', 'Project Management Specialists (##)']


**Load findings**
- Header rows sit at different depths: row 8 in the SOC structure, row 9 in the SOC crosswalk, row 6 in the hybrid file (above them are title and note rows). The CIP-SOC workbook has a clean header on every sheet.
- The NCES CSVs wrap every code in an Excel formula (`="01.0101"`) so Excel keeps leading zeros. Strip `="` before using a code. Read as text, the zeros survive.
- The hybrid file pads 889 cells (titles and notes only; no code cell is padded). Strip anyway.
- 166 SOC crosswalk titles carry footnote marks `(#)` / `(##)` (e.g. *Managers, All Other (#)* → *Project Management Specialists (##)*). They sit on codes that take part in a split or merge. Strip them from the title; the relation in §2 carries the same information.
- The CIP 2020 list (2,848 rows) stacks 2-digit families (50), 4-digit series (473) and 6-digit codes (2,325, including old codes that moved or were deleted).

## 2. What one row is (grain)

**Presumed keys:** structure `code`; SOC crosswalk `soc2010 × soc2018`; hybrid `oes2019 × soc2018 × oes2018 × soc2010`; CIP crosswalk `CIPCode2010 × CIPCode2020`; CIP-SOC `CIP2020Code × SOC2018Code`.

In [5]:
checks = {
    "SOC structure: code": st.duplicated("code").sum(),
    "SOC 2010→2018: soc2010 × soc2018": socxw.duplicated(["soc2010", "soc2018"]).sum(),
    "OEWS hybrid: all four codes": hyb.duplicated(["oes2019", "soc2018", "oes2018", "soc2010"]).sum(),
    "CIP 2010→2020: code pair": cipxw.duplicated(["CIPCode2010", "CIPCode2020"]).sum(),
    "CIP-SOC: code pair": cipsoc["CIP-SOC"].duplicated(["CIP2020Code", "SOC2018Code"]).sum(),
    "SOC-CIP: code pair": cipsoc["SOC-CIP"].duplicated(["SOC2018Code", "CIP2020Code"]).sum(),
}
print("Duplicate keys:", checks)

# How SOC 2010 and SOC 2018 codes relate: 1:1, splits (1:m), merges (m:1), or tangles (m:m)
n18 = socxw.groupby("soc2010").soc2018.transform("nunique")
n10 = socxw.groupby("soc2018").soc2010.transform("nunique")
# split = one 2010 code becomes several 2018 codes; merge = several 2010 codes become one 2018 code
socxw["relation"] = (n18.gt(1).astype(int) + 2 * n10.gt(1).astype(int)).map({0: "1:1", 1: "split", 2: "merge", 3: "split + merge"})
print("\nSOC 2010 → 2018 pairs by relation:", socxw.relation.value_counts().to_dict(),
      f"| SOC 2010 codes: {socxw.soc2010.nunique()} | SOC 2018 codes: {socxw.soc2018.nunique()}")
display(socxw[socxw.relation != "1:1"].sort_values(["relation", "soc2010"]).groupby("relation").head(3)[["relation", "soc2010", "title2010", "soc2018", "title2018"]])

Duplicate keys: {'SOC structure: code': np.int64(0), 'SOC 2010→2018: soc2010 × soc2018': np.int64(0), 'OEWS hybrid: all four codes': np.int64(0), 'CIP 2010→2020: code pair': np.int64(0), 'CIP-SOC: code pair': np.int64(0), 'SOC-CIP: code pair': np.int64(0)}

SOC 2010 → 2018 pairs by relation: {'1:1': 766, 'split': 70, 'split + merge': 32, 'merge': 32} | SOC 2010 codes: 840 | SOC 2018 codes: 867


,relation,soc2010,title2010,soc2018,title2018
100,merge,15-2091,Mathematical Technicians,15-2099,"Mathematical Science Occupations, All Other (##)"
272,merge,25-4021,Librarians,25-4022,Librarians and Media Collections Specialists (##)
274,merge,25-9011,Audio-Visual and Multimedia Collections Specialists,25-4022,Librarians and Media Collections Specialists (##)
6,split,11-2031,Public Relations and Fundraising Managers (#),11-2032,Public Relations Managers
7,split,11-2031,Public Relations and Fundraising Managers (#),11-2033,Fundraising Managers
8,split,11-3011,Administrative Services Managers (#),11-3012,Administrative Services Managers
38,split + merge,11-9199,"Managers, All Other (#)",13-1082,Project Management Specialists (##)
57,split + merge,13-1199,"Business Operations Specialists, All Other (#)",13-1082,Project Management Specialists (##)
65,split + merge,13-2051,Financial Analysts (#),13-2054,Financial Risk Specialists (##)


In [6]:
# CIP 2010 -> 2020: one row per detailed code; does any 2010 code go to more than one 2020 code?
cdet = cipxw[cipxw.CIPCode2010.str.fullmatch(CIP) | cipxw.CIPCode2020.str.fullmatch(CIP)]
print("CIP crosswalk, detailed rows by Action:", cdet.Action.value_counts().to_dict())
print("CIP 2010 codes mapping to more than one 2020 code:", (cdet[cdet.CIPCode2010 != ""].groupby("CIPCode2010").CIPCode2020.nunique() > 1).sum())
print("CIP 2020 codes receiving more than one 2010 code:", (cdet[cdet.CIPCode2010 != ""].groupby("CIPCode2020").CIPCode2010.nunique() > 1).sum())
display(cdet[cdet.Action.isin(["Moved to", "Deleted"])].groupby("Action").head(4)[["CIPCode2010", "CIPTitle2010", "Action", "CIPCode2020"]])

# CIP-SOC: links per major and per job (real links only)
cs = cipsoc["CIP-SOC"]
real = cs[(cs.SOC2018Code != "99-9999") & (cs.CIP2020Code != "99.9999")]
per_cip, per_soc = real.groupby("CIP2020Code").size(), real.groupby("SOC2018Code").size()
print(f"\nCIP-SOC real links: {len(real):,} | majors: {per_cip.size:,} | jobs: {per_soc.size:,}")
display(pd.DataFrame({"jobs per major": per_cip.describe(percentiles=[.5, .9]), "majors per job": per_soc.describe(percentiles=[.5, .9])}).round(1).T)
top = per_soc.sort_values(ascending=False).head(8).rename("majors").to_frame().join(real.drop_duplicates("SOC2018Code").set_index("SOC2018Code").SOC2018Title)
top

CIP crosswalk, detailed rows by Action: {'No substantive changes': 1568, 'New': 460, 'Moved to': 145, 'Deleted': 7}
CIP 2010 codes mapping to more than one 2020 code: 0
CIP 2020 codes receiving more than one 2010 code: 0


,CIPCode2010,CIPTitle2010,Action,CIPCode2020
25,01.0309,Viticulture and Enology.,Moved to,01.1004
606,15.0503,Energy Management and Systems Technology/Technician.,Moved to,15.1701
607,15.0505,Solar Energy Technology/Technician.,Moved to,15.1703
787,19.0000,Work and Family Studies.,Moved to,19.1001
2364,60.0406,Clinical Cytogenetics Residency Program.,Deleted,60.0406
2366,60.0408,Clinical Molecular Genetics Residency Program.,Deleted,60.0408
2393,60.0499,"Medical Residency Programs - General Certificates, Other.",Deleted,60.0499
2462,60.0568,Physical Medicine and Rehabilitation/Psychiatry Residency Program.,Deleted,60.0568



CIP-SOC real links: 5,723 | majors: 1,948 | jobs: 687


,count,mean,std,min,50%,90%,max
jobs per major,1948.0,2.9,1.9,1.0,3.0,5.0,23.0
majors per job,687.0,8.3,20.0,1.0,3.0,17.0,337.0


,majors,SOC2018Title
SOC2018Code,,
25-1071,337,"Health Specialties Teachers, Postsecondary"
19-1042,210,"Medical Scientists, Except Epidemiologists"
11-9121,149,Natural Sciences Managers
11-9199,128,"Managers, All Other"
25-2031,92,"Secondary School Teachers, Except Special and Career/Technical Edu..."
25-1124,88,"Foreign Language and Literature Teachers, Postsecondary"
27-3091,87,Interpreters and Translators
25-1042,83,"Biological Science Teachers, Postsecondary"


**Grain findings**
- **Every presumed key is unique** in all six tables.
- SOC 2010 → 2018 is not one-to-one. Of 900 pairs, 766 are 1:1; 70 are splits (e.g. `11-2031` *Public Relations and Fundraising Managers* → `11-2032` + `11-2033`), 32 are merges (`25-4021` *Librarians* + `25-9011` → `25-4022`) and 32 are both (`11-9199` and `13-1199` → `13-1082` *Project Management Specialists*). So moving a count from SOC 2010 to SOC 2018 sometimes has to split it, which needs a weight. That matters for OEWS 2018 and the 2016-26 / 2018-28 projections (§6).
- CIP 2010 → 2020 is strictly one-to-one for detailed codes: no 2010 code goes to two 2020 codes, and no 2020 code receives two 2010 codes. 1,568 unchanged, 145 moved to a new number, 7 deleted (all `60.xxxx` medical residencies, out of scope), 460 new. Recoding IPEDS 2015–2019 is a plain lookup with no splitting.
- CIP→SOC is many-to-many with a long tail. 5,723 real links between 1,948 majors and 687 jobs. A typical major links to 3 jobs (max 23, *Business Administration*), and a typical job to 3 majors. The tail is the problem: `25-1071` *Health Specialties Teachers, Postsecondary* links to 337 majors, `19-1042` *Medical Scientists* to 210, `11-9121` *Natural Sciences Managers* to 149, `11-9199` *Managers, All Other* to 128. Counting "graduates per job" by summing every linked major would pile thousands of graduates onto these catch-all jobs (§6d).

## 3. Column profile

In [7]:
pat = lambda s: s.str.replace(r"\d", "9", regex=True).value_counts().to_dict()
print("SOC 2010→2018 code shapes:", pat(socxw.soc2010), pat(socxw.soc2018))
print("OEWS hybrid code shapes:", {c: pat(hyb[c]) for c in ["oes2019", "soc2018", "oes2018", "soc2010"]})
print("OEWS hybrid NOTES:", hyb.notes.replace("", "(blank: same code in every column)").value_counts().to_dict())
print("CIP-SOC code shapes:", pat(cs.CIP2020Code), pat(cs.SOC2018Code))
print("CIP 2020 list, Action:", cip20.Action.value_counts().to_dict(), "| code shapes:", pat(cip20.CIPCode))
print("SOC structure levels:", st.level.value_counts().to_dict())

SOC 2010→2018 code shapes: {'99-9999': 900} {'99-9999': 900}
OEWS hybrid code shapes: {'oes2019': {'99-9999': 868}, 'soc2018': {'99-9999': 868}, 'oes2018': {'99-9999': 868}, 'soc2010': {'99-9999': 868}}
OEWS hybrid NOTES: {'(blank: same code in every column)': 677, 'Hybrid (code & title)': 73, 'Title change only': 48, 'Code change only': 28, '2018 SOC broad occupation': 27, 'Code and title change only': 7, 'OES-specific aggregation': 6, 'Hybrid (code & title). OES-specific code under 2010 SOC': 2}
CIP-SOC code shapes: {'99.9999': 6097} {'99-9999': 6097}
CIP 2020 list, Action: {'No substantive changes': 1994, 'New': 544, 'Moved from': 149, 'Moved to': 149, 'Deleted': 12} | code shapes: {'99.9999': 2325, '99.99': 473, '99': 50}
SOC structure levels: {'detailed': 867, 'broad': 459, 'minor': 98, 'major': 23}


**Profile findings**
- Every code in every file has the right shape (`NN-NNNN` / `NN.NNNN`). No blanks in code columns.
- **Hybrid file `NOTES`:** 677 rows have the same code in every column (no change). The rest say what kind of change: `Hybrid (code & title)` (73 rows), `Title change only` (48), `Code change only` (28), `2018 SOC broad occupation` (27: OEWS published a broad code where it couldn't split the detail yet), `OES-specific aggregation` (6), `Code and title change only` (7), and 2 hybrids built on an OES-specific 2010 code.
- The SOC structure has exactly four levels: 23 major, 98 minor, 459 broad, 867 detailed.

## 4. Codes, hierarchy and special values

(a) Which codes are "real" SOC 2018 codes? The structure file is the authority. A code that isn't in it is either a SOC 2010 code, a BLS aggregate (hybrid), or a typo. (b) The CIP-SOC "no match" rows: both directions use a sentinel (`99-9999` / `99.9999`, title *NO MATCH*).

In [8]:
S18 = dict(zip(st.code, st.level))
DETAILED18 = set(st.code[st.level == "detailed"])
print("SOC crosswalk 2018 side = structure's detailed codes:", set(socxw.soc2018) == DETAILED18)

hyb_codes = (set(hyb.oes2019) | set(hyb.oes2018)) - set(S18) - set(socxw.soc2010)
print(f"Codes in the hybrid file that are neither SOC 2018 nor SOC 2010: {len(hyb_codes)}")
h = hyb[hyb.oes2019.isin(hyb_codes)].groupby(["oes2019", "oes2019_title"]).agg(
    soc2018_parts=("soc2018", lambda s: ", ".join(sorted(set(s)))), notes=("notes", "first")).reset_index()
display(h)

SOC crosswalk 2018 side = structure's detailed codes: True
Codes in the hybrid file that are neither SOC 2018 nor SOC 2010: 30


,oes2019,oes2019_title,soc2018_parts,notes
0,11-9198,"Personal Service Managers, All Other; Entertainment and Recreation...","11-9072, 11-9179, 11-9199",Hybrid (code & title)
1,13-1198,Project Management Specialists and Business Operations Specialists...,"13-1082, 13-1199",Hybrid (code & title)
2,13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and...","13-2051, 13-2054, 13-2099",Hybrid (code & title)
3,15-1245,Database Administrators and Architects,"15-1242, 15-1243",Hybrid (code & title)
4,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,"15-1252, 15-1253",Hybrid (code & title)
5,15-1257,Web Developers and Digital Interface Designers,"15-1254, 15-1255",Hybrid (code & title)
6,15-2098,"Data Scientists and Mathematical Science Occupations, All Other","15-2051, 15-2099",Hybrid (code & title)
7,17-3098,Calibration Technologists and Technicians and Engineering Technolo...,"17-3028, 17-3029",Hybrid (code & title)
8,19-4045,Geological and Hydrologic Technicians,"19-4043, 19-4044",Hybrid (code & title)
9,21-1018,"Substance Abuse, Behavioral Disorder, and Mental Health Counselors","21-1011, 21-1014",OES-specific aggregation


In [9]:
nomatch_cip = cs[cs.SOC2018Code == "99-9999"]
nomatch_soc = cs[cs.CIP2020Code == "99.9999"]
print(f"CIP-SOC sheet: {len(nomatch_cip)} majors with no job (99-9999) | {len(nomatch_soc)} jobs with no major (99.9999)")
print("Same as the 'Unmatched CIP Codes' / 'Unmatched SOC Codes' sheets:",
      set(nomatch_cip.CIP2020Code) == set(cipsoc["Unmatched CIP Codes"].CIP2020Code),
      set(nomatch_soc.SOC2018Code) == set(cipsoc["Unmatched SOC Codes"].SOC2018Code))
a = set(zip(cs.CIP2020Code, cs.SOC2018Code)); b = set(zip(cipsoc["SOC-CIP"].CIP2020Code, cipsoc["SOC-CIP"].SOC2018Code))
print(f"Pairs only in CIP-SOC: {len(a - b)} | only in SOC-CIP: {len(b - a)}")
display(pd.DataFrame(sorted(a ^ b), columns=["CIP2020Code", "SOC2018Code"]).assign(
    in_sheet=lambda d: [("CIP-SOC" if (c, s) in a else "SOC-CIP") for c, s in zip(d.CIP2020Code, d.SOC2018Code)]).head(12))
print("No-match jobs, by SOC major group:", nomatch_soc.SOC2018Code.str[:2].value_counts().head(8).to_dict())

CIP-SOC sheet: 194 majors with no job (99-9999) | 180 jobs with no major (99.9999)
Same as the 'Unmatched CIP Codes' / 'Unmatched SOC Codes' sheets: True True
Pairs only in CIP-SOC: 4 | only in SOC-CIP: 0


,CIP2020Code,SOC2018Code,in_sheet
0,03.0204,11-9199,CIP-SOC
1,03.0204,19-3011,CIP-SOC
2,03.0204,19-4061,CIP-SOC
3,03.0204,25-1063,CIP-SOC


No-match jobs, by SOC major group: {'51': 48, '53': 24, '43': 21, '47': 20, '35': 10, '39': 10, '55': 9, '41': 7}


**Code findings**
- 30 codes in the hybrid file are neither SOC 2018 nor SOC 2010: 27 used for OEWS May 2019–2020, plus 3 used only in May 2018 (`25-3098`, `51-2098`, `53-1048`). Each lists its SOC 2018 parts (e.g. `15-1256` = `15-1252` Software Developers + `15-1253` QA Analysts).
- Correction to notebook 04 and the source map: `21-1018` *Substance Abuse, Behavioral Disorder, and Mental Health Counselors* is not a SOC 2018 code. It's a BLS OES-specific aggregation of `21-1011` + `21-1014`. Notebook 04 named it as an example of a "real SOC 2018 code ending in 8". The rule still stands (a trailing 8 is not a hybrid marker: `29-1218` *Obstetricians and Gynecologists* is real SOC 2018), but the only reliable test is the structure file and the hybrid list.
- **The CIP-SOC "no match" sentinels:** 194 majors have no job (`99-9999`) and 180 jobs have no major (`99.9999`, title *NO MATCH*). They're identical to the *Unmatched* sheets. The no-major jobs are mostly production (48), transport (24), office (21) and construction (20): jobs reached by training or experience, not a college major.
- The two directions of the crosswalk differ by 4 links: `03.0204` *Environmental/Natural Resource Economics* → 4 jobs appears in the `CIP-SOC` sheet but not in `SOC-CIP`. Use `CIP-SOC` as the master.
- "Moved to" means opposite things in the two NCES files. In the 2010→2020 crosswalk, the row for the *old* code says `Moved to`. In the CIP 2020 list, the *new* code says `Moved to` and the old code says `Moved from` (`01.0309` *Moved from* → `01.1004` *Moved to*). Filtering the CIP 2020 list on `Moved to` to drop old codes removes 145 live codes instead. Use the crosswalk's code pair, never the label alone. Also, `Deleted` rows in the crosswalk still repeat the old code in the 2020 column, so a lookup must check `Action` before trusting the target.

## 5. Coverage

Do the bridges cover the whole of each code list?

In [10]:
# In this list "Moved from" marks the OLD code and "Moved to" the NEW one (the 2010→2020 crosswalk labels its old-code rows "Moved to")
print(cip20[cip20.CIPCode.isin(["01.0309", "01.1004"])][["CIPCode", "Action", "CIPTitle"]].to_string(index=False))
ACTIVE20 = set(cip20.CIPCode[cip20.CIPCode.str.fullmatch(CIP) & ~cip20.Action.isin(["Moved from", "Deleted"])])
in_cs = set(cs.CIP2020Code) - {"99.9999"}
print(f"Active CIP 2020 detailed codes: {len(ACTIVE20):,} | in the CIP-SOC file: {len(ACTIVE20 & in_cs):,} | missing: {len(ACTIVE20 - in_cs)} | extra: {len(in_cs - ACTIVE20)}")
miss = cip20[cip20.CIPCode.isin(ACTIVE20 - in_cs)]
print("Missing codes by Action:", miss.Action.value_counts().to_dict(), "| by 2-digit family:", miss.CIPCode.str[:2].value_counts().to_dict())
display(miss[["CIPCode", "CIPTitle"]].head(8))
print(f"SOC 2018 detailed codes: {len(DETAILED18)} | in the CIP-SOC file: {len(DETAILED18 & set(cs.SOC2018Code))} "
      f"| non-structure SOC codes in CIP-SOC: {sorted(set(cs.SOC2018Code) - DETAILED18 - {'99-9999'})}")
cip10_det = set(cdet.CIPCode2010[cdet.CIPCode2010 != ""])
print(f"CIP 2010 detailed codes in the CIP crosswalk: {len(cip10_det):,} (every one has a 2020 target: {cdet[cdet.CIPCode2010 != ''].CIPCode2020.ne('').all()})")

CIPCode     Action                 CIPTitle
01.0309 Moved from Viticulture and Enology.
01.1004   Moved to Viticulture and Enology.
Active CIP 2020 detailed codes: 2,173 | in the CIP-SOC file: 2,142 | missing: 31 | extra: 0
Missing codes by Action: {'New': 31} | by 2-digit family: {'55': 12, '51': 6, '01': 3, '16': 2, '52': 2, '12': 1, '21': 1, '28': 1, '29': 1, '44': 1, '49': 1}


,CIPCode,CIPTitle
30,01.0480,Reserved.
48,01.0680,Reserved.
79,01.1180,Reserved.
364,12.0580,Reserved.
799,16.1702,Reserved.
800,16.1799,Reserved.
855,21.0101,Reserved.
1095,28.0801,Reserved.


SOC 2018 detailed codes: 867 | in the CIP-SOC file: 867 | non-structure SOC codes in CIP-SOC: []
CIP 2010 detailed codes in the CIP crosswalk: 1,720 (every one has a 2020 target: True)


**Coverage findings**
- CIP→SOC covers every live CIP 2020 major except 30 codes titled *Reserved.* (placeholders with no programs) and one new code, `49.0109` *Remote Aircraft Pilot*. No CIP-SOC code is outside the CIP 2020 list.
- CIP→SOC covers all 867 SOC 2018 detailed jobs (687 with a major, 180 marked *no match*), and uses no code outside the SOC 2018 structure.
- The SOC crosswalk covers all 840 SOC 2010 and all 867 SOC 2018 detailed codes.
- CIP 2010→2020 covers all 1,720 CIP 2010 detailed codes.

## 6. Relationships: will every source make it through?

**(a) Jobs.** Collect every SOC-style code in the NYC slices that notebooks 02–04 explored, then classify each against the references above.

In [11]:
codes = {}
for f in man[man.key.str.startswith("nysdol_projections")].file:
    d = pd.read_csv(RAW / f, dtype=str, keep_default_na=False)
    d = d[d.Area == "New York City"]
    codes[f"projections {d.Period.iloc[0].replace(' ', '')}"] = set(d.SOC.str.lstrip("'"))
for f in man[man.key == "bls_oews_metro"].file:
    z = zipfile.ZipFile(RAW / f)
    msa = next(n for n in z.namelist() if Path(n).name.startswith("MSA_"))
    d = pd.read_excel(io.BytesIO(z.read(msa)), dtype=str, keep_default_na=False, usecols=lambda c: c.strip().upper() in ("AREA", "OCC_CODE"))
    d.columns = [c.strip().upper() for c in d.columns]
    codes[f"OEWS NY metro {2000 + int(Path(f).stem[4:6])}"] = set(d.OCC_CODE[d.AREA.str.strip() == "35620"].str.strip())
w = pd.read_csv(RAW / man[man.key == "nysdol_oews_regions"].file.iloc[0], dtype=str, keep_default_na=False)
w.columns = [c.strip() for c in w.columns]
w = w.apply(lambda s: s.str.strip())
w_nyc = w[w["Area Name"] == "New York City"]
codes["NYS DOL NYC wages"] = set(w_nyc["Standard Occupational Code"])
ep = pd.read_excel(RAW / man[man.key == "bls_ep_education"].file.iloc[0], sheet_name="Table 5.4", header=1, dtype=str, keep_default_na=False)
codes["BLS EP education"] = set(ep.iloc[:, 1][ep.iloc[:, 1].str.fullmatch(SOC)])
codes = {k: {c for c in v if pd.Series([c]).str.fullmatch(SOC).iloc[0]} for k, v in sorted(codes.items())}

SOC10 = set(socxw.soc2010)
def kind(c):
    if c == "00-0000": return "total"
    if c in DETAILED18: return "SOC 2018 detailed"
    if c in S18: return "SOC 2018 group"
    if c in hyb_codes: return "BLS aggregate (hybrid file)"
    if c in SOC10: return "SOC 2010 detailed"
    if c.endswith("00"): return "SOC 2010 group (by pattern)"
    return "unknown"

rows = [{"source": k, "code": c, "kind": kind(c)} for k, v in codes.items() for c in v]
allc = pd.DataFrame(rows)
order = ["SOC 2018 detailed", "SOC 2018 group", "BLS aggregate (hybrid file)", "SOC 2010 detailed", "SOC 2010 group (by pattern)", "unknown", "total"]
pd.crosstab(allc.source, allc.kind, margins=True, margins_name="all codes")[[c for c in order if c in allc.kind.unique()] + ["all codes"]]

kind,SOC 2018 detailed,SOC 2018 group,BLS aggregate (hybrid file),SOC 2010 detailed,SOC 2010 group (by pattern),total,all codes
source,,,,,,,
BLS EP education,819,7,4,1,0,0,831
NYS DOL NYC wages,668,28,4,1,0,1,702
OEWS NY metro 2018,666,28,6,72,0,1,773
OEWS NY metro 2019,694,35,27,2,0,1,759
OEWS NY metro 2020,691,35,27,2,0,1,756
OEWS NY metro 2021,769,29,4,1,0,1,804
OEWS NY metro 2022,765,29,4,1,0,1,800
OEWS NY metro 2023,768,29,4,1,0,1,803
OEWS NY metro 2024,759,29,4,1,0,1,794


In [12]:
# Resolve every code to the SOC 2018 detailed jobs it covers
desc = {}   # group code -> its detailed descendants
for lvl in ["major", "minor", "broad"]:
    for g, d in st[st.level == "detailed"].groupby(f"in_{lvl}"):
        desc[g] = set(d.code)
x10 = socxw.groupby("soc2010").soc2018.agg(set).to_dict()
h19 = hyb.groupby("oes2019").soc2018.agg(set).to_dict()
h18 = hyb.groupby("oes2018").soc2010.agg(set).to_dict()     # 2018-only aggregates resolve through their SOC 2010 parts

def resolve(c, depth=0):
    if depth > 3: return set()
    if c in DETAILED18: return {c}
    if c in desc: return desc[c]
    if c in h19 and c not in SOC10: return set().union(*(resolve(p, depth + 1) for p in h19[c]))
    if c in x10: return x10[c]
    if c in h18: return set().union(*(resolve(p, depth + 1) for p in h18[c]))
    if c.endswith("00"):   # SOC 2010 group code: every SOC 2010 detailed code under it
        stem = c.rstrip("0")
        return set().union(*(x10[k] for k in x10 if k.startswith(stem))) if any(k.startswith(stem) for k in x10) else set()
    return set()

allc["resolves_to"] = allc.code.map(lambda c: len(resolve(c)))
allc["path"] = allc.kind.where(allc.resolves_to > 0, "UNRESOLVED")
bad = allc[(allc.resolves_to == 0) & (allc.kind != "total")]
print("Codes that reach no SOC 2018 job:", bad.drop_duplicates("code")[["code", "kind"]].values.tolist() or "none")
print("Detailed jobs per non-detailed code (median, max):",
      allc[allc.kind.isin(["BLS aggregate (hybrid file)", "SOC 2010 detailed"])].drop_duplicates("code").resolves_to.agg(["median", "max"]).to_dict())
show = ["15-1256", "13-2098", "15-1257", "13-1198", "15-2098", "21-1018", "25-2052", "15-1132", "25-3098", "15-1100", "31-1000"]
pd.DataFrame({"kind": [kind(c) for c in show], "SOC 2018 detailed jobs": [", ".join(sorted(resolve(c))) for c in show]}, index=show)

Codes that reach no SOC 2018 job: none
Detailed jobs per non-detailed code (median, max): {'median': 1.5, 'max': 8.0}


,kind,SOC 2018 detailed jobs
15-1256,BLS aggregate (hybrid file),"15-1252, 15-1253"
13-2098,BLS aggregate (hybrid file),"13-2051, 13-2054, 13-2099"
15-1257,BLS aggregate (hybrid file),"15-1254, 15-1255"
13-1198,BLS aggregate (hybrid file),"13-1082, 13-1199"
15-2098,BLS aggregate (hybrid file),"15-2051, 15-2099"
21-1018,BLS aggregate (hybrid file),"21-1011, 21-1014"
25-2052,SOC 2010 detailed,"25-2055, 25-2056"
15-1132,SOC 2010 detailed,"15-1252, 15-1253"
25-3098,BLS aggregate (hybrid file),25-3099
15-1100,SOC 2010 group (by pattern),"13-1082, 15-1211, 15-1212, 15-1221, 15-1231, 15-1232, 15-1241, 15-..."


(b) BLS aggregates still in use. Which non-SOC codes appear in the *current* files (OEWS 2025, NYS DOL wages, BLS EP)? These need the bridge permanently, not only for 2019–20:

In [13]:
current = ["OEWS NY metro 2025", "NYS DOL NYC wages", "BLS EP education"]
persist = allc[allc.source.isin(current) & allc.kind.isin(["BLS aggregate (hybrid file)", "SOC 2010 detailed"])]
p = persist.groupby("code").source.agg(lambda s: ", ".join(sorted(s))).to_frame("in current files")
p["kind"] = p.index.map(kind)
p["title (hybrid file)"] = p.index.map(hyb.drop_duplicates("oes2019").set_index("oes2019").oes2019_title)
p["SOC 2018 parts"] = p.index.map(lambda c: ", ".join(sorted(resolve(c))))
p

,in current files,kind,title (hybrid file),SOC 2018 parts
code,,,,
21-1018,"BLS EP education, NYS DOL NYC wages, OEWS NY metro 2025",BLS aggregate (hybrid file),"Substance Abuse, Behavioral Disorder, and Mental Health Counselors","21-1011, 21-1014"
25-2052,"BLS EP education, NYS DOL NYC wages, OEWS NY metro 2025",SOC 2010 detailed,"Special Education Teachers, Kindergarten and Elementary School","25-2055, 25-2056"
25-9045,"BLS EP education, NYS DOL NYC wages, OEWS NY metro 2025",BLS aggregate (hybrid file),"Teaching Assistants, Except Postsecondary","25-9042, 25-9043, 25-9049"
51-2028,"BLS EP education, NYS DOL NYC wages, OEWS NY metro 2025",BLS aggregate (hybrid file),"Electrical, Electronic, and Electromechanical Assemblers, Except C...","51-2022, 51-2023"
53-1047,"BLS EP education, NYS DOL NYC wages, OEWS NY metro 2025",BLS aggregate (hybrid file),First-Line Supervisors of Transportation and Material Moving Worke...,"53-1042, 53-1043, 53-1044, 53-1049"


**(c) Majors.** Run NYC's IPEDS CIP codes (notebook 05: NYC colleges from each year's directory, detailed rows) through CIP 2010→2020 and then CIP→SOC:

In [14]:
def first_csv(zp, prefer_rv=True):
    names = [n for n in zipfile.ZipFile(zp).namelist() if n.lower().endswith(".csv")]
    rv = [n for n in names if "_rv" in n.lower()]
    return (rv or names)[0]

def read_zip_csv(zp, **kw):
    raw = zipfile.ZipFile(zp).read(first_csv(zp))
    try: text = raw.decode("utf-8-sig")
    except UnicodeDecodeError: text = raw.decode("latin-1")
    d = pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False, **kw)
    d.columns = [c.strip().upper() for c in d.columns]
    return d.apply(lambda s: s.str.strip())

ip = []
for y in range(2015, 2025):
    hd = read_zip_csv(RAW / f"graduates/ipeds_directory/HD{y}.zip", usecols=lambda c: c.strip().upper() in ("UNITID", "COUNTYCD"))
    nyc_ids = set(hd.UNITID[hd.COUNTYCD.isin(NYC_COUNTIES)])
    c = read_zip_csv(RAW / f"graduates/ipeds_completions/C{y}_A.zip", usecols=lambda c: c.strip().upper() in ("UNITID", "CIPCODE", "MAJORNUM", "AWLEVEL", "CTOTALT"))
    c = c[c.UNITID.isin(nyc_ids) & c.CIPCODE.str.fullmatch(CIP) & c.MAJORNUM.eq("1")]
    ip.append(c.assign(year=y, awards=pd.to_numeric(c.CTOTALT), level=c.AWLEVEL.astype(int)))
ip = pd.concat(ip, ignore_index=True)
ip = ip[ip.level.isin([3, 5])]          # v1 scope: associate's (3) and bachelor's (5)

to20 = dict(zip(cdet.CIPCode2010, cdet.CIPCode2020))
ip["cip2020"] = ip.CIPCODE.where(ip.year >= 2020, ip.CIPCODE.map(to20))
linked = set(real.CIP2020Code)
ip["status"] = "linked to a job"
ip.loc[ip.cip2020.isin(set(nomatch_cip.CIP2020Code)), "status"] = "NO MATCH in crosswalk"
ip.loc[~ip.cip2020.isin(linked | set(nomatch_cip.CIP2020Code)), "status"] = "code not in crosswalk"
ip.loc[ip.cip2020.isna(), "status"] = "CIP 2010 code not mapped"
t = ip.pivot_table(index="year", columns="status", values="awards", aggfunc="sum", fill_value=0)
t["share linked"] = (t["linked to a job"] / t.sum(axis=1)).map("{:.1%}".format)
display(t)
print("CIP 2010 codes in NYC 2015-19 that changed code in 2020:",
      ip[(ip.year < 2020) & (ip.CIPCODE != ip.cip2020)].groupby(["CIPCODE", "cip2020"]).awards.sum().sort_values(ascending=False).head(6).to_dict())

status,NO MATCH in crosswalk,linked to a job,share linked
year,,,
2015,1971,76863,97.5%
2016,2156,78480,97.3%
2017,1885,78881,97.7%
2018,1858,80336,97.7%
2019,2070,83305,97.6%
2020,2161,83989,97.5%
2021,2238,84592,97.4%
2022,2346,79627,97.1%
2023,2102,72884,97.2%


CIP 2010 codes in NYC 2015-19 that changed code in 2020: {('39.0606', '38.0207'): 4712, ('43.0106', '43.0406'): 425, ('43.0117', '43.0405'): 250, ('51.3817', '51.3203'): 197, ('51.0808', '01.8301'): 128, ('43.0116', '43.0403'): 2}


(d) How much of the major→job link runs through postsecondary teaching? The crosswalk links almost every academic major to its *postsecondary teacher* job (`25-1xxx`). For associate's and bachelor's graduates that link is not a realistic first job:

In [15]:
r24 = ip[ip.year == 2024].groupby(["cip2020", "level"]).awards.sum().reset_index()
jobs_of = real.groupby("CIP2020Code").SOC2018Code.agg(set)
r24["jobs"] = r24.cip2020.map(jobs_of)
r24 = r24.dropna(subset=["jobs"])
r24["only_teacher_jobs"] = r24.jobs.map(lambda s: all(j.startswith("25-1") for j in s))
r24["links"] = r24.jobs.map(len)
r24["teacher_links"] = r24.jobs.map(lambda s: sum(j.startswith("25-1") for j in s))
print(f"2024 NYC associate + bachelor's awards with a linked job: {r24.awards.sum():,}")
print(f"  ...whose ONLY linked jobs are postsecondary teaching: {r24.awards[r24.only_teacher_jobs].sum():,} ({r24.awards[r24.only_teacher_jobs].sum() / r24.awards.sum():.1%})")
print(f"  share of all major→job links that point to postsecondary teaching (award-weighted): "
      f"{(r24.awards * r24.teacher_links / r24.links).sum() / r24.awards.sum():.1%}")
names = cip20.set_index("CIPCode").CIPTitle
r24[r24.only_teacher_jobs].groupby("cip2020").awards.sum().sort_values(ascending=False).head(8).to_frame().join(names)

2024 NYC associate + bachelor's awards with a linked job: 71,137
  ...whose ONLY linked jobs are postsecondary teaching: 6,478 (9.1%)
  share of all major→job links that point to postsecondary teaching (award-weighted): 28.4%


,awards,CIPTitle
cip2020,,
24.0101,3439,Liberal Arts and Sciences/Liberal Studies.
38.0207,1333,Talmudic Studies.
38.0206,593,Jewish/Judaic Studies.
38.0101,222,Philosophy.
50.9999,216,"Visual and Performing Arts, Other."
23.1404,125,English Literature (British and Commonwealth).
05.0299,80,"Ethnic, Cultural Minority, Gender, and Group Studies, Other."
16.0104,57,Comparative Literature.


(e) And from the job side: how much NYC employment sits in jobs that no major leads to? (NYS DOL NYC wage table, detailed jobs, hybrids resolved.)

In [16]:
wd = w_nyc[w_nyc["Standard Occupational Code"].map(kind).isin(["SOC 2018 detailed", "BLS aggregate (hybrid file)"])].copy()
wd["employment"] = pd.to_numeric(wd.Employment.str.replace(",", ""), errors="coerce")
print("Wage rows with no employment figure:", wd.employment.isna().sum())
wd["has_major"] = wd["Standard Occupational Code"].map(lambda c: bool(resolve(c) & set(real.SOC2018Code)))
print(f"NYC wage jobs checked: {len(wd)} | with at least one major: {wd.has_major.mean():.1%} of jobs, "
      f"{wd.employment[wd.has_major].sum() / wd.employment.sum():.1%} of employment")
wd[~wd.has_major].sort_values("employment", ascending=False)[["Standard Occupational Code", "Occupational Title", "employment"]].head(10)

Wage rows with no employment figure: 117


NYC wage jobs checked: 672 | with at least one major: 81.8% of jobs, 79.7% of employment


,Standard Occupational Code,Occupational Title,employment
4993,37-2011,"Janitors and Cleaners, Except Maids and Housekeeping Cleaners",96390.0
5030,41-2031,Retail Salespersons,95100.0
4984,35-3031,Waiters and Waitresses,70150.0
4983,35-3023,Fast Food and Counter Workers,66090.0
5027,41-2011,Cashiers,58470.0
5283,53-7065,Stockers and Order Fillers,49440.0
5280,53-7062,"Laborers and Freight, Stock, and Material Movers, Hand",35820.0
4981,35-2021,Food Preparation Workers,31080.0
4994,37-2012,Maids and Housekeeping Cleaners,22210.0
4986,35-9011,Dining Room and Cafeteria Attendants and Bartender Helpers,17490.0


**(f) Decision D1 check.** D1 fills the broken 2022-32 codes with BLS national growth and openings rates. Now that `occupation.xlsx` is downloaded (2025-35 edition), does BLS EP have rates for every SOC 2018 part of those codes?

In [17]:
occ = pd.read_excel(RAW / man[man.key == "bls_ep_occupation"].file.iloc[0], sheet_name="Table 1.2", header=1, dtype=str, keep_default_na=False)
occ.columns = [c.strip() for c in occ.columns]
occ = occ[occ.iloc[:, 1].str.fullmatch(SOC)]
code_col, type_col = occ.columns[1], occ.columns[2]
print("Table 1.2 rows by Occupation type:", occ[type_col].value_counts().to_dict())
d1 = ["15-1256", "13-2098", "15-1257", "13-1198", "15-2098"]
ep_codes = set(occ[code_col])
rate_cols = [c for c in occ.columns if c.startswith("Employment change, percent") or c.startswith("Occupational openings")]
out = []
for c in d1:
    for part in sorted(resolve(c)):
        r = occ[occ[code_col] == part]
        out.append({"broken code": c, "SOC 2018 part": part, "in BLS EP 2025-35": part in ep_codes,
                    **({k: r[k].iloc[0] for k in rate_cols} if len(r) else {})})
pd.DataFrame(out)

Table 1.2 rows by Occupation type: {'Line item': 831, 'Summary': 281}


,broken code,SOC 2018 part,in BLS EP 2025-35,"Employment change, percent, 2025–35","Occupational openings, 2025–35 annual average"
0,15-1256,15-1252,True,10.2,95.3
1,15-1256,15-1253,True,5.7,10.7
2,13-2098,13-2051,True,7.2,24.7
3,13-2098,13-2054,True,7.4,4.8
4,13-2098,13-2099,True,4.1,9.4
5,15-1257,15-1254,True,3.8,5.3
6,15-1257,15-1255,True,6,8.3
7,13-1198,13-1082,True,6.7,76.5
8,13-1198,13-1199,True,3.9,98.7
9,15-2098,15-2051,True,34.6,24.8


**Join findings**
- (a) Every job code in every source reaches SOC 2018. Across 14 NYC slices (4 projection editions, 8 OEWS years, NYS DOL wages, BLS EP), 10,389 code uses: 9,288 are SOC 2018 detailed, 748 are SOC 2018 groups, and the rest resolve through the bridges: 132 BLS aggregates, 204 SOC 2010 codes, and 4 SOC 2010 group codes (`15-1100`, `31-1000` in the 2016-26 and 2018-28 projections, resolved through their SOC 2010 detailed members). Which bridge a source needs depends on its year:
  - **SOC 2010 → 2018**: OEWS 2018 (72 codes), projections 2016-26 (60) and 2018-28 (57).
  - **Hybrids**: OEWS 2019–2020 (27), projections 2020-30 (22) and 2022-32 (14).
  - A non-detailed code covers 1–8 SOC 2018 jobs (median 1.5).
- (b) The bridge is permanent, not a 2019–20 patch. Five non-SOC-2018 codes are still in the current files (OEWS 2025, NYS DOL wages, BLS EP 2025): `21-1018`, `25-9045`, `51-2028`, `53-1047`, and `25-2052` (a SOC 2010 code BLS kept instead of splitting it into `25-2055` / `25-2056`). The job dimension needs these as real rows, with a bridge to their SOC 2018 parts.
- **(c) Majors:** every NYC associate/bachelor's CIP code in 2015–2024 maps to CIP 2020 and is in the crosswalk. 97.1%–97.7% of awards link to at least one job every year; the rest are *no match* majors, mostly general and "other" codes (`51.0000` *Health Services/Allied Health, General*, `30.0000` *Multi-/Interdisciplinary Studies*, `…99` codes; notebook 05 §6). The biggest 2010→2020 recode in NYC is `39.0606` → `38.0207` *Talmudic Studies* (4,712 awards 2015–19): an old code that moved family.
- (d) The major→job links lean heavily on postsecondary teaching. Award-weighted, 28% of the links from NYC's 2024 associate and bachelor's awards point to a `25-1xxx` *postsecondary teacher* job, which needs a doctorate or master's. For 6,478 awards (9.1%), postsecondary teaching is the *only* link. By far the largest is `24.0101` *Liberal Arts and Sciences* (3,439 awards). So the link has to be filtered by degree level: a job counts for a major only if the major's award level meets the job's typical entry education (notebook 04). Without that filter, liberal arts graduates "supply" postsecondary teachers. This is a core decision, to be logged in the decision log.
- **(e) Job side:** 81.8% of NYC's detailed wage jobs (672) have at least one linked major. Those jobs are 79.7% of NYC employment (counting only the 555 jobs whose employment NYS DOL publishes; 117 are suppressed). The jobs with no major are the large low-education service jobs (janitors 96k, retail salespersons 95k, waiters 70k, fast food 66k). That's expected: they're demand no degree feeds.
- (f) Decision D1 can run as written. All 11 SOC 2018 parts of the five broken 2022-32 codes (`15-1256`, `13-2098`, `15-1257`, `13-1198`, `15-2098`) have national growth and openings rates in BLS EP Table 1.2. The edition is 2025-35, not the 2024-34 named in D1.

## 7. Tie-out

The SOC Policy Committee publishes the size of SOC 2018 (23 major groups, 98 minor groups, 459 broad occupations, 867 detailed occupations) and SOC 2010 (840 detailed). NCES's CIP files carry their own change counts. Compare:

In [18]:
pub = {"SOC 2018 major": 23, "SOC 2018 minor": 98, "SOC 2018 broad": 459, "SOC 2018 detailed": 867, "SOC 2010 detailed": 840}
got = {**{f"SOC 2018 {k}": v for k, v in st.level.value_counts().items()}, "SOC 2010 detailed": socxw.soc2010.nunique()}
display(pd.DataFrame({"published": pub, "file": {k: got[k] for k in pub}}).assign(match=lambda d: d.published == d.file))

# CIP: the 2010->2020 file and the CIP 2020 list should agree on what was new, moved or deleted (detailed codes)
a = cdet.Action.value_counts()
b = cip20[cip20.CIPCode.str.fullmatch(CIP)].Action.value_counts()
pd.DataFrame({"CIP 2010→2020 crosswalk": a, "CIP 2020 list": b}).fillna(0).astype(int)

,published,file,match
SOC 2018 major,23,23,True
SOC 2018 minor,98,98,True
SOC 2018 broad,459,459,True
SOC 2018 detailed,867,867,True
SOC 2010 detailed,840,840,True


,CIP 2010→2020 crosswalk,CIP 2020 list
Action,,
Deleted,7,7
Moved from,0,145
Moved to,145,145
New,460,460
No substantive changes,1568,1568


**Tie-out findings**
- The SOC structure matches the published size exactly (23 / 98 / 459 / 867), and the crosswalk holds all 840 SOC 2010 detailed codes.
- The two CIP files agree on every change count: 460 new, 145 moved, 7 deleted, 1,568 unchanged. The CIP 2020 list also has 145 `Moved from` rows: the old codes, labelled from the other side (§4).

## 8. Fitness for purpose

| Question | Bridge | Fit | Why |
|---|---|---|---|
| Put every job source on one job list (SOC 2018) | Structure + SOC 2010→2018 + hybrid file | Use | Every code in every NYC slice resolves (§6a). The five permanent aggregates stay as their own job rows |
| Recode IPEDS 2015–2019 majors to CIP 2020 | CIP 2010→2020 | Use | Strictly one-to-one for detailed codes. A lookup, no weights |
| Major → job ("what can I do with this major?", brief S4) | CIP 2020→SOC 2018 | for listing jobs | Covers every live major and every job |
| Graduates per job / per opening (the gap) | CIP→SOC | Use with rules | Many-to-many with catch-all jobs (337 majors → one teacher job). Needs (1) the degree-level filter (§6d), and (2) an allocation rule for majors with several jobs (equal split vs. weighted by NYC employment or ACS PUMS outcomes). Both get logged as decisions |
| Moving a SOC 2010 count onto SOC 2018 | SOC 2010→2018 | partial | 102 of 900 pairs split or tangle, so a count has to be shared out with a weight (e.g. OEWS 2021 employment shares) |
| Where graduates actually work | — | no | The crosswalk is expert judgment ("prepares for"), not observed outcomes. ACS PUMS (notebook 08) tests it |

## 9. Issues list and proposed staging tables

| # | Issue | Source · rows affected | Proposed treatment (decided in staging/core) |
|---|---|---|---|
| 1 | Header rows at different depths; title rows above | BLS xlsx · 3 files | Read with the header row from §1; assert column names |
| 2 | Codes wrapped in `="…"` | NCES CSVs · all | Strip `="`; assert pattern |
| 3 | Padded titles/notes | Hybrid file · 889 cells | Strip all text |
| 4 | Footnote marks `(#)`, `(##)` in titles | SOC crosswalk · 166 titles | Strip from titles; relation column carries the meaning |
| 5 | SOC 2010→2018 splits and merges | 102 of 900 pairs | `relation` column; split counts with an employment weight (decide in core) |
| 6 | BLS aggregate codes not in SOC 2018 | 30 codes; 5 still current | Job dimension gets them as `is_bls_aggregate` rows + a bridge to SOC 2018 parts |
| 7 | SOC 2010 group codes in old projections | `15-1100`, `31-1000` | Resolve through SOC 2010 detailed members |
| 8 | `Moved to` means opposite things in the two NCES files | CIP list · 290 rows | Use the crosswalk's code pair; never filter on the label alone |
| 9 | `Deleted` rows repeat the old code as the 2020 code | CIP crosswalk · 7 rows | Null the target when `Action = Deleted` |
| 10 | `99-9999` / `99.9999` no-match sentinels mixed into the link sheet | CIP-SOC · 374 rows | Keep as flags on the major and job dimensions, not as links |
| 11 | `SOC-CIP` sheet misses 4 links | `03.0204` | Use `CIP-SOC` as master |
| 12 | Catch-all jobs with hundreds of majors; postsecondary teaching links | 28% of award-weighted links | **Degree-level filter** + allocation rule → decision log (core) |
| 13 | Crosswalk is expert judgment, not outcomes | all links | Label in the dashboard ("prepares for"); compare with ACS PUMS in nb 08 |
| 14 | Notebook 04 / source map named `21-1018` as a real SOC 2018 code | docs | Corrected here; fix the wording in nb 04 and the source map |

**Proposed staging tables** (reference data, loaded as-is, one table per file):
- **`stg_bls__soc_2018_structure`**: key `soc_code`; `soc_title`, `level`, `major_code`, `minor_code`, `broad_code`. 1,447 rows.
- **`stg_bls__soc_2010_2018`**: key `soc_2010 × soc_2018`; titles (marks stripped), `relation`. 900 rows.
- **`stg_bls__oews_hybrid`**: key `oes_2019 × soc_2018 × oes_2018 × soc_2010`; titles, `notes`. 868 rows.
- **`stg_nces__cip_2010_2020`**: key `cip_2010` (detailed only); `cip_2020` (null when deleted), `action`, titles. 1,720 rows with a 2010 code + 460 new.
- **`stg_nces__cip_2020`**: key `cip_code`; `title`, `level` (2/4/6-digit), `action`, `definition`. 2,848 rows.
- **`stg_nces__cip_soc`**: key `cip_2020 × soc_2018`, real links only (5,723 rows); the no-match lists become flags (`has_job_link`, `has_major_link`).

In core, these become one job dimension (SOC 2018 detailed + the 5 permanent aggregates), a `bridge_job_code` (any source code → SOC 2018 detailed, with a weight), and a `bridge_major_job` (CIP 2020 → SOC 2018 after the degree-level filter).